### Chargement du dataset

In [1]:
import torchvision.transforms as transforms

train_transform = transforms.Compose([
 transforms.Resize(256),
 transforms.RandomCrop(224),
 transforms.RandomHorizontalFlip(),
 transforms.ColorJitter(brightness=0.2, contrast=0.2),
 transforms.ToTensor(),
 transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
 transforms.Resize(256),
 transforms.CenterCrop(224),
 transforms.ToTensor(),
 transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from pathlib import Path

# Définition des chemins
DATA_DIR = Path('../data/raw/Rock Data')
TRAIN_DIR = DATA_DIR / 'train'
VALID_DIR = DATA_DIR / 'valid'
TEST_DIR = DATA_DIR / 'test'

# Création des datasets avec les transformations de la Phase 2
train_ds = ImageFolder(TRAIN_DIR, transform=train_transform)
valid_ds = ImageFolder(VALID_DIR, transform=val_transform)
test_ds = ImageFolder(TEST_DIR, transform=val_transform)

# Création des DataLoaders (num_workers=0 pour stabilité sous Windows)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

print(f"Datasets chargés avec succès !")
print(f"Classes détectées : {train_ds.classes}")

Datasets chargés avec succès !
Classes détectées : ['Basalt', 'Clay', 'Conglomerate', 'Diatomite', 'Shale-(Mudstone)', 'Siliceous-sinter', 'chert', 'gypsum', 'olivine-basalt']


### Architecture ResNet-50 fine-tunée

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
# Charger ResNet-50 pré-entraîné sur ImageNet
model = models.resnet50(weights='IMAGENET1K_V1')
# Geler les couches (optionnel — phase 1 d'entraînement)
for param in model.parameters():
 param.requires_grad = False
# Remplacer la dernière couche FC pour 9 classes
num_features = model.fc.in_features # 2048
model.fc = nn.Sequential(
 nn.Dropout(0.5),
 nn.Linear(num_features, 9)
)


### Configuration de l'entraînement

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
scheduler = StepLR(optimizer, step_size=7, gamma=0.1)

### Boucle d'entraînement

#### Phase A (5 epochs, tout gelé sauf FC)

In [ ]:
# ── PHASE A : Entraîner uniquement la couche FC ──
for param in model.parameters():
    param.requires_grad = False
model.fc.requires_grad_(True)

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

EPOCHS_A = 5
best_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print("=" * 50)
print("PHASE A — Entraînement de la couche FC uniquement")
print("=" * 50)

for epoch in range(EPOCHS_A):
    # Entraînement
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    val_acc = 100. * correct / total
    history['train_loss'].append(train_loss / len(train_loader))
    history['val_loss'].append(val_loss / len(valid_loader))
    history['val_acc'].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'resnet50_best.pth')

    print(f'Epoch {epoch+1}/{EPOCHS_A} — '
          f'Train Loss: {train_loss/len(train_loader):.4f} — '
          f'Val Loss: {val_loss/len(valid_loader):.4f} — '
          f'Val Acc: {val_acc:.2f}%')

print(f'\nMeilleure accuracy Phase A : {best_acc:.2f}%')

####  Phase B (15 epochs, Bloc3 + Bloc4 + FC dégelés)

In [ ]:
# ── PHASE B : Dégeler layer3, layer4 et FC ──
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
model.fc.requires_grad_(True)

optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 0.0001},
    {'params': model.layer4.parameters(), 'lr': 0.0001},
    {'params': model.fc.parameters(),     'lr': 0.0001},
])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

EPOCHS_B = 15

print("=" * 50)
print("PHASE B — Fine-tuning layer3 + layer4 + FC")
print("=" * 50)

for epoch in range(EPOCHS_B):
    # Entraînement
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    val_acc = 100. * correct / total
    history['train_loss'].append(train_loss / len(train_loader))
    history['val_loss'].append(val_loss / len(valid_loader))
    history['val_acc'].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'resnet50_best.pth')

    print(f'Epoch {epoch+1}/{EPOCHS_B} — '
          f'Train Loss: {train_loss/len(train_loader):.4f} — '
          f'Val Loss: {val_loss/len(valid_loader):.4f} — '
          f'Val Acc: {val_acc:.2f}%')

print(f'\nMeilleure accuracy Phase B : {best_acc:.2f}%')
print(f'Modèle sauvegardé : resnet50_best.pth')